In [1]:
from equiv_dens.scripts import transform_orbitals
from equiv_dens.utils.orbitals import orbitals_from_hamiltonian, parse_orbitals
import numpy as np
import pyscf
from equiv_dens.data.hamiltonian_dataset import HamiltonianDataset
from pyscf import gto, dft, df, lib
from pyscf.scf import hf
from equiv_dens.utils import base as utils
from pyscf.dft import numint
from equiv_dens.utils import grids
import torch

%load_ext autoreload
%autoreload 2

/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


Use "numpy" for Fourier Transform


In [2]:
pyscf_orb = np.load('datasets/h2o_pbe-def2svp_4999_pyscf_augccpvqzjkfit_10.db.npy', allow_pickle=True)
svp_orb_db = HamiltonianDataset('datasets/h2o_pbe-def2svp_4999.db')

In [21]:
pyscf_orb = np.load('datasets/ethanol_pbe-def2svp_30000_pyscf_augccpvqzjkfit_10.db.npy', allow_pickle=True)
svp_orb_db = HamiltonianDataset('datasets/ethanol_pbe-def2svp_30000.db')

In [22]:
svp_orb = svp_orb_db.collate_fn(list(range(len(pyscf_orb))))

In [23]:
print(svp_orb_db, 'svp_orb_db')
print('full ham shape', svp_orb['full_hamiltonian'].shape)

<equiv_dens.data.hamiltonian_dataset.HamiltonianDataset object at 0x7f7b80356898> svp_orb_db
full ham shape torch.Size([10, 72, 72])


In [24]:
#compare svp npy vs db hamiltonians

for i in range(len(svp_orb['full_hamiltonian'][0, 0])):
    print('i', i)
    print('npy ham', svp_orb['full_hamiltonian'][0, i])
    print('db ham', svp_orb_db.collate_fn([0])['full_hamiltonian'][0, i])

i 0
npy ham tensor([-9.9495e+00, -3.4606e+00, -1.6414e+00, -1.2160e-03,  3.4100e-04,
        -2.3690e-03, -1.0900e-04,  2.6000e-05, -2.9400e-04,  3.4000e-05,
         2.7000e-04,  5.0000e-06, -6.6000e-05,  1.4100e-04, -0.0000e+00,
        -1.0958e-01, -4.9466e-01,  1.0202e-01,  6.9088e-02, -5.1904e-02,
         8.0211e-01,  5.4320e-01, -4.0810e-01,  5.8955e-02, -7.8474e-02,
         1.1427e-02,  3.9926e-02,  4.2941e-02,  0.0000e+00, -1.5679e-02,
        -3.2038e-01,  1.9100e-03, -4.3000e-03,  1.6018e-02,  9.5140e-02,
        -2.1413e-01,  7.9763e-01, -6.0510e-03,  1.6240e-03,  1.2744e-02,
         1.3621e-02, -2.5009e-02, -5.1456e-01, -9.2468e-01, -7.0053e-01,
         4.2122e-01,  1.1547e-01, -4.0075e-01, -8.5975e-01, -8.6936e-02,
        -5.1502e-01, -2.6614e-01, -2.5280e-03, -1.8942e-01,  1.3800e-04,
         1.6100e-04, -2.1200e-04, -5.4000e-03, -2.3800e-01,  6.1500e-04,
         7.2900e-04,  7.5000e-05, -2.3290e-03, -1.8480e-01,  2.5200e-04,
         4.3000e-05, -6.7000e-05, -1.65

In [25]:
hamiltonians_svp = svp_orb['full_hamiltonian']
overlap_svp = svp_orb['overlap_matrix']

orbitals_svp, ens = orbitals_from_hamiltonian(hamiltonians_svp, overlap_svp)

In [26]:
H = overlap_svp[0].T @ orbitals_svp[0] @ np.diag(ens[0]) @ np.linalg.inv(orbitals_svp[0])
print('H', H[0,:])
print('H orig', hamiltonians_svp[0, 0,:])


H tensor([-9.9495e+00, -3.4606e+00, -1.6414e+00, -1.2159e-03,  3.4103e-04,
        -2.3691e-03, -1.0902e-04,  2.5954e-05, -2.9389e-04,  3.4092e-05,
         2.7000e-04,  4.8773e-06, -6.5996e-05,  1.4081e-04,  4.6566e-09,
        -1.0958e-01, -4.9466e-01,  1.0202e-01,  6.9088e-02, -5.1904e-02,
         8.0211e-01,  5.4320e-01, -4.0810e-01,  5.8955e-02, -7.8474e-02,
         1.1427e-02,  3.9926e-02,  4.2941e-02,  2.0443e-07, -1.5679e-02,
        -3.2038e-01,  1.9100e-03, -4.2999e-03,  1.6018e-02,  9.5140e-02,
        -2.1413e-01,  7.9763e-01, -6.0510e-03,  1.6239e-03,  1.2744e-02,
         1.3621e-02, -2.5009e-02, -5.1456e-01, -9.2468e-01, -7.0054e-01,
         4.2122e-01,  1.1546e-01, -4.0075e-01, -8.5976e-01, -8.6936e-02,
        -5.1502e-01, -2.6614e-01, -2.5281e-03, -1.8942e-01,  1.3817e-04,
         1.6079e-04, -2.1215e-04, -5.4000e-03, -2.3800e-01,  6.1499e-04,
         7.2906e-04,  7.4980e-05, -2.3289e-03, -1.8480e-01,  2.5209e-04,
         4.3177e-05, -6.6667e-05, -1.6572e-02, -3

In [27]:
print(orbitals_svp[0][[0], :])
print(pyscf_orb[0][1]['mo_coeff'][[0], :])
atom_types = utils.numbers_to_symbols(svp_orb_db.database.Z)

print('')
basis_def = np.load('datasets/def2svp_orbital_basis.npy', allow_pickle=True).item()
split_svp = parse_orbitals(orbitals_svp[0][[0], :], atom_types, basis_def)
split_pyscf = parse_orbitals(pyscf_orb[0][1]['mo_coeff'][[0], :], atom_types, basis_def)
for i in range(len(split_svp)):
    print('svp', split_svp[i])
    print('pyscf', split_pyscf[i])
    print('')

[[-4.67188329e-05  9.87978756e-01  1.56519189e-03  8.16508904e-02
  -1.30793452e-01 -1.93848953e-01 -4.97091413e-02  1.54424263e-02
  -2.46629827e-02  3.84674378e-04  1.69265028e-02 -8.72889906e-03
  -1.43514830e-03  4.71717119e-02  1.06796168e-01 -1.33804813e-01
  -1.49030583e-02 -7.18464702e-02  5.33423685e-02  1.72328595e-02
   3.72410491e-02 -1.34009961e-02 -1.17171176e-01  6.09233938e-02
   1.16692651e-02  1.19134262e-01  2.45913770e-02 -2.35559996e-02
  -1.28760546e-01 -8.18484947e-02 -4.36717607e-02 -2.16145404e-02
   5.80947623e-02  1.46312967e-01 -5.33481799e-02 -1.09498380e-02
  -8.40622745e-03 -2.56414320e-02  1.56885702e-02 -4.29751799e-02
  -9.18442011e-03 -1.97471976e-02  2.50176545e-02  7.42643431e-04
  -2.31999578e-03 -1.75471464e-03  3.29128504e-02 -5.28962463e-02
  -5.64568723e-03  1.26922000e-02  2.23037228e-02  3.94516662e-02
   1.28278192e-02 -1.51107740e-03 -8.92859697e-03 -3.83836329e-02
   2.79838592e-02  1.62113309e-02  5.20149209e-02 -8.98101330e-02
   6.15055

In [28]:
import copy
from equiv_dens.scripts.transform_hamiltonians import transform
%load_ext autoreload
%autoreload 2

svp_orb_201 = copy.deepcopy(svp_orb)
svp_orb_210 = copy.deepcopy(svp_orb)

print('svp_old', svp_orb['full_hamiltonian'][0, 0, :])
svp_orb_201['full_hamiltonian'] = transform(svp_orb_201['full_hamiltonian'], atom_types, 'def2-SVP_to_pyscf_201')
svp_orb_210['full_hamiltonian'] = transform(svp_orb_210['full_hamiltonian'], atom_types, 'def2-SVP_to_pyscf_210')
svp_orb_201['overlap_matrix'] = transform(svp_orb_201['overlap_matrix'], atom_types, 'def2-SVP_to_pyscf_201')
svp_orb_210['overlap_matrix'] = transform(svp_orb_210['overlap_matrix'], atom_types, 'def2-SVP_to_pyscf_210')
print('svp_old after', svp_orb['full_hamiltonian'][0, 0, :])
print('svp_new after', svp_orb_201['full_hamiltonian'][0, 0, :])
    
#np.save('datasets/h2o_dynamic_ham_def2svp_201_dft_f.npy', svp_orb_201, allow_pickle=True)
#np.save('datasets/h2o_dynamic_ham_def2svp_210_dft_f.npy', svp_orb_210, allow_pickle=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
svp_old tensor([-9.9495e+00, -3.4606e+00, -1.6414e+00, -1.2160e-03,  3.4100e-04,
        -2.3690e-03, -1.0900e-04,  2.6000e-05, -2.9400e-04,  3.4000e-05,
         2.7000e-04,  5.0000e-06, -6.6000e-05,  1.4100e-04, -0.0000e+00,
        -1.0958e-01, -4.9466e-01,  1.0202e-01,  6.9088e-02, -5.1904e-02,
         8.0211e-01,  5.4320e-01, -4.0810e-01,  5.8955e-02, -7.8474e-02,
         1.1427e-02,  3.9926e-02,  4.2941e-02,  0.0000e+00, -1.5679e-02,
        -3.2038e-01,  1.9100e-03, -4.3000e-03,  1.6018e-02,  9.5140e-02,
        -2.1413e-01,  7.9763e-01, -6.0510e-03,  1.6240e-03,  1.2744e-02,
         1.3621e-02, -2.5009e-02, -5.1456e-01, -9.2468e-01, -7.0053e-01,
         4.2122e-01,  1.1547e-01, -4.0075e-01, -8.5975e-01, -8.6936e-02,
        -5.1502e-01, -2.6614e-01, -2.5280e-03, -1.8942e-01,  1.3800e-04,
         1.6100e-04, -2.1200e-04, -5.4000e-03, -2.3800e-01,  6.1500e-04,
         7.2900e-04,  7.5000

In [29]:

hamiltonians_svp = svp_orb['full_hamiltonian']
overlap_svp = svp_orb['overlap_matrix']
orbitals_svp, ens = orbitals_from_hamiltonian(hamiltonians_svp, overlap_svp)

hamiltonians_svp_210 = svp_orb_210['full_hamiltonian']
overlap_svp_210 = svp_orb_210['overlap_matrix']
orbitals_svp_210, ens_210 = orbitals_from_hamiltonian(hamiltonians_svp_210, overlap_svp_210)


hamiltonians_svp_201 = svp_orb_201['full_hamiltonian']
overlap_svp_201 = svp_orb_201['overlap_matrix']
orbitals_svp_201, ens_201 = orbitals_from_hamiltonian(hamiltonians_svp_201, overlap_svp_201)

print('hamiltonian', hamiltonians_svp[0, 0, :])
print('hamiltonian 210', hamiltonians_svp_210[0, 0, :])
print('hamiltonian 201', hamiltonians_svp_201[0, 0, :])

print('overlap', overlap_svp[0, 0, :])
print('overlap 210', overlap_svp_210[0, 0, :])
print('overlap 201', overlap_svp_201[0, 0, :])

print('orbitals', orbitals_svp[0][0, :])
print('orbitals 210', orbitals_svp_210[0][0, :])
print('orbitals 201', orbitals_svp_201[0][0, :])

hamiltonian tensor([-9.9495e+00, -3.4606e+00, -1.6414e+00, -1.2160e-03,  3.4100e-04,
        -2.3690e-03, -1.0900e-04,  2.6000e-05, -2.9400e-04,  3.4000e-05,
         2.7000e-04,  5.0000e-06, -6.6000e-05,  1.4100e-04, -0.0000e+00,
        -1.0958e-01, -4.9466e-01,  1.0202e-01,  6.9088e-02, -5.1904e-02,
         8.0211e-01,  5.4320e-01, -4.0810e-01,  5.8955e-02, -7.8474e-02,
         1.1427e-02,  3.9926e-02,  4.2941e-02,  0.0000e+00, -1.5679e-02,
        -3.2038e-01,  1.9100e-03, -4.3000e-03,  1.6018e-02,  9.5140e-02,
        -2.1413e-01,  7.9763e-01, -6.0510e-03,  1.6240e-03,  1.2744e-02,
         1.3621e-02, -2.5009e-02, -5.1456e-01, -9.2468e-01, -7.0053e-01,
         4.2122e-01,  1.1547e-01, -4.0075e-01, -8.5975e-01, -8.6936e-02,
        -5.1502e-01, -2.6614e-01, -2.5280e-03, -1.8942e-01,  1.3800e-04,
         1.6100e-04, -2.1200e-04, -5.4000e-03, -2.3800e-01,  6.1500e-04,
         7.2900e-04,  7.5000e-05, -2.3290e-03, -1.8480e-01,  2.5200e-04,
         4.3000e-05, -6.7000e-05, -1.65

In [30]:
import scipy
for i in range(len(pyscf_orb)):
    #print('pyscf_orb en', np.diag(pyscf_orb[i][1]['mo_energy']).shape)
    #print(pyscf_orb[0][0])
    mol = gto.M(atom=pyscf_orb[i][0]['atom'], basis=pyscf_orb[i][0]['basis'], unit=pyscf_orb[i][0]['unit'])
    mol.build()
    ovlp = hf.get_ovlp(mol)
    #print('ovlp py', ovlp[23, :])
    #print('ovlp db', overlap_svp[i,23,:])
    #print('ovlp 210', overlap_svp_210[i, 23, :])
    #print('ovlp 201', overlap_svp_201[i, 23, :])
    print(np.allclose(ovlp, overlap_svp_201[i], atol=1e-2))
    #print('ovlp', ovlp[23])
    #print('ovlp 201', overlap_svp_201[i, 23])
    H = ovlp @ pyscf_orb[i][1]['mo_coeff'] @ np.diag(pyscf_orb[i][1]['mo_energy']) @ np.linalg.inv(pyscf_orb[i][1]['mo_coeff'])
    H_201 = overlap_svp_201[i] @ orbitals_svp_201[i] @ np.diag(ens[i]) @ np.linalg.inv(orbitals_svp_201[i])
    #print('H', H[0])
    #H = ovlp @ pyscf_orb[i][1]['mo_coeff'].T @ np.diag(pyscf_orb[i][1]['mo_energy']) @ pyscf_orb[i][1]['mo_coeff']
    print('H', H[0])
    print('H 201', hamiltonians_svp_201[i, 0])
    print('H 201', H_201[0])
    print(np.allclose(H, hamiltonians_svp_201[i], atol=1e-2))
    #print('ovlp I', ovlp * scipy.linalg.inv(ovlp))
    print('orbs\n', np.stack([pyscf_orb[i][1]['mo_coeff'][0].T, orbitals_svp_201[i][0].T], axis=1))
    #print('orb 201', orbitals_svp_201[i][0])
   

True
H [-9.94960867e+00 -3.46061332e+00 -1.64147021e+00 -2.36373748e-03
 -1.21184311e-03  3.39912095e-04 -2.92606021e-04 -1.08330022e-04
  2.60050207e-05  3.60883494e-05  2.71443106e-04  2.90261970e-06
 -6.92516783e-05  1.45777543e-04 -4.58528628e-07 -1.09581034e-01
 -4.94664746e-01 -5.19044157e-02  1.02017427e-01  6.90884953e-02
 -4.08107248e-01  8.02119303e-01  5.43208337e-01  5.89547876e-02
 -7.84740227e-02  1.14266701e-02  3.99256484e-02  4.29410376e-02
  2.74626985e-08 -1.56787896e-02 -3.20384035e-01  1.60181370e-02
  1.91025287e-03 -4.30003472e-03  7.97638007e-01  9.51417890e-02
 -2.14130202e-01 -6.05070787e-03  1.62447080e-03  1.27443471e-02
  1.36206712e-02 -2.50085179e-02 -5.14570082e-01 -9.24691023e-01
  1.15467308e-01 -7.00544804e-01  4.21225830e-01 -4.00759327e-01
 -8.59768436e-01 -2.66148605e-01 -8.69371608e-02 -5.15028608e-01
 -2.52826365e-03 -1.89425380e-01 -2.11709638e-04  1.37869487e-04
  1.60679830e-04 -5.39956986e-03 -2.37999478e-01  7.46377018e-05
  6.14584201e-04  

In [31]:
print(orbitals_svp[0][[0], :])
print(pyscf_orb[0][1]['mo_coeff'][[0], :])
atom_types = utils.numbers_to_symbols(svp_orb_db.database.Z)
basis_def = np.load('datasets/def2svp_orbital_basis.npy', allow_pickle=True).item()
print('')
split_svp_210 = parse_orbitals(orbitals_svp_210[0][[0], :], atom_types, basis_def)

split_svp_201 = parse_orbitals(orbitals_svp_201[0][[0], :], atom_types, basis_def)
split_pyscf = parse_orbitals(pyscf_orb[0][1]['mo_coeff'][[0], :], atom_types, basis_def)
for i in range(len(split_svp)):
    print('svp', split_svp[i])
    print('svp 210', split_svp_210[i])
    print('svp 201', split_svp_201[i])
    print('pyscf', split_pyscf[i])
    print('')

[[-4.67188329e-05  9.87978756e-01  1.56519189e-03  8.16508904e-02
  -1.30793452e-01 -1.93848953e-01 -4.97091413e-02  1.54424263e-02
  -2.46629827e-02  3.84674378e-04  1.69265028e-02 -8.72889906e-03
  -1.43514830e-03  4.71717119e-02  1.06796168e-01 -1.33804813e-01
  -1.49030583e-02 -7.18464702e-02  5.33423685e-02  1.72328595e-02
   3.72410491e-02 -1.34009961e-02 -1.17171176e-01  6.09233938e-02
   1.16692651e-02  1.19134262e-01  2.45913770e-02 -2.35559996e-02
  -1.28760546e-01 -8.18484947e-02 -4.36717607e-02 -2.16145404e-02
   5.80947623e-02  1.46312967e-01 -5.33481799e-02 -1.09498380e-02
  -8.40622745e-03 -2.56414320e-02  1.56885702e-02 -4.29751799e-02
  -9.18442011e-03 -1.97471976e-02  2.50176545e-02  7.42643431e-04
  -2.31999578e-03 -1.75471464e-03  3.29128504e-02 -5.28962463e-02
  -5.64568723e-03  1.26922000e-02  2.23037228e-02  3.94516662e-02
   1.28278192e-02 -1.51107740e-03 -8.92859697e-03 -3.83836329e-02
   2.79838592e-02  1.62113309e-02  5.20149209e-02 -8.98101330e-02
   6.15055

In [32]:

for i in range(len(pyscf_orb)):
    mol_dict, coeff_dict = pyscf_orb[i]
    mol = gto.Mole(**mol_dict)

    if not mol._built:
        # build_start = time.time()
        mol.build()
        # print('build time', time.time() - build_start)
    # ao_start = time.time()
    atom_numbers = mol.atom_charges()[None,:]
    pos = mol.atom_coords(unit='Angstrom')[None, :]
    atoms = {'atom_numbers': atom_numbers}
    grid_spec = grids.spherical_grid(atoms)
    
    sample_coords, coord_weights = grids.spherical_radial_sampling(grid_spec, 10000000,
                                                    atom_numbers,
                                                    torch.tensor(pos))
    
    
    
    scaled_sample_coords = sample_coords.detach().cpu().numpy() * utils.to_bohr  # convert Angstrom grid to Bohr
    ao = numint.eval_ao(mol, scaled_sample_coords[0])
    # print('ao time', time.time() - ao_start)
    # rho_start = time.time()
    rho_pyscf = numint.eval_rho2(mol, ao, mo_occ=coeff_dict['mo_occ'], mo_coeff=coeff_dict['mo_coeff'])
    print('dens int', np.sum(rho_pyscf * coord_weights.numpy())) 
    rho_201 = numint.eval_rho2(mol, ao, mo_occ=coeff_dict['mo_occ'], mo_coeff=orbitals_svp_201[i])
    rho_210 = numint.eval_rho2(mol, ao, mo_occ=coeff_dict['mo_occ'], mo_coeff=orbitals_svp_210[i])
    rho_orc = numint.eval_rho2(mol, ao, mo_occ=coeff_dict['mo_occ'], mo_coeff=orbitals_svp[i])
    print('dens_diff 201', np.sum(np.abs(rho_pyscf - rho_201) * coord_weights.numpy())/np.sum(rho_pyscf * coord_weights.numpy()))
    print('dens_diff 210', np.sum(np.abs(rho_pyscf - rho_210) * coord_weights.numpy())/np.sum(rho_pyscf * coord_weights.numpy()))
    print('dens_diff orc', np.sum(np.abs(rho_pyscf - rho_orc) * coord_weights.numpy())/np.sum(rho_pyscf * coord_weights.numpy()))

dens int 26.00001743720385
dens_diff 201 1.0965487258815996e-05
dens_diff 210 0.16692111821589473
dens_diff orc 0.21601850989574295
dens int 25.9999936030681
dens_diff 201 1.1037232207511221e-05
dens_diff 210 0.13790176874537607
dens_diff orc 0.19470443277027166
dens int 26.000003763191483
dens_diff 201 1.0632361940757583e-05
dens_diff 210 0.16736312892506516
dens_diff orc 0.2089019559532733
dens int 26.000028734525266
dens_diff 201 1.2861136605917995e-05
dens_diff 210 0.16179874764174465
dens_diff orc 0.1994131885400157
dens int 26.000031540360496
dens_diff 201 1.0402235260208988e-05
dens_diff 210 0.1607781908833682
dens_diff orc 0.2099407991822518
dens int 26.00006267915532
dens_diff 201 1.0564647562124327e-05
dens_diff 210 0.13844608043879522
dens_diff orc 0.18200626632177486
dens int 26.000008479885352
dens_diff 201 1.0354430841421263e-05
dens_diff 210 0.14180448043481714
dens_diff orc 0.18784224861451362
dens int 26.000013879294638
dens_diff 201 1.1837145623849881e-05
dens_diff 21

In [35]:
pyscf_orb = np.load('datasets/ethanol_pbe-def2svp_30000_pyscf_augccpvqzjkfit_10.db.npy', allow_pickle=True)
pyscf_db_orb = np.load('datasets/ethanol_pbe-def2svp_30000_pyscf.npy', allow_pickle=True)

In [43]:
pyscf_orb = np.load('datasets/h2o_pbe-def2svp_4999_pyscf_augccpvqzjkfit_10.db.npy', allow_pickle=True)
pyscf_db_orb = np.load('datasets/h2o_pbe-def2svp_4999_pyscf.npy', allow_pickle=True)

In [42]:

for i in range(len(pyscf_orb)):
    mol_dict, coeff_dict = pyscf_orb[i]
    mol = gto.Mole(**mol_dict)

    if not mol._built:
        # build_start = time.time()
        mol.build()
        # print('build time', time.time() - build_start)
    # ao_start = time.time()
    atom_numbers = mol.atom_charges()[None,:]
    pos = mol.atom_coords(unit='Angstrom')[None, :]
    atoms = {'atom_numbers': atom_numbers}
    grid_spec = grids.spherical_grid(atoms)
    
    sample_coords, coord_weights = grids.spherical_radial_sampling(grid_spec, 10000000,
                                                    atom_numbers,
                                                    torch.tensor(pos))
    
    
    
    scaled_sample_coords = sample_coords.detach().cpu().numpy() * utils.to_bohr  # convert Angstrom grid to Bohr
    ao = numint.eval_ao(mol, scaled_sample_coords[0])
    # print('ao time', time.time() - ao_start)
    # rho_start = time.time()
    rho_pyscf = numint.eval_rho2(mol, ao, mo_occ=coeff_dict['mo_occ'], mo_coeff=coeff_dict['mo_coeff'])
    print('dens int', np.sum(rho_pyscf * coord_weights.numpy())) 
    rho_201 = numint.eval_rho2(mol, ao, mo_occ=pyscf_db_orb[i][1]['mo_occ'], mo_coeff=pyscf_db_orb[i][1]['mo_coeff'])
    print('dens_diff 201', np.sum(np.abs(rho_pyscf - rho_201) * coord_weights.numpy())/np.sum(rho_pyscf * coord_weights.numpy()))
    mol_df = gto.mole.Mole(**mol_dict)
    mol_df.basis = coeff_dict['auxbasis'] 
    mol_df.build()
        # print('build time', time.time() - build_start)
    # df_coeff = np.concatenate(self.density_fitting['df_coeffs'][i])
    df_coeff = coeff_dict['df_coeff'] 
    # ao_start = time.time()
    ao = numint.eval_ao(mol_df, scaled_sample_coords[0])
    # print('ao time', time.time() - ao_start)
    # rho_start = time.time()
    rho_df = np.einsum('ij,j->i', ao, df_coeff)
    rho_df_201 = np.einsum('ij,j->i', ao, pyscf_db_orb[i][1]['df_coeff'])
    print('dens_diff df', np.sum(np.abs(rho_pyscf - rho_df) * coord_weights.numpy())/np.sum(rho_pyscf * coord_weights.numpy()))
    print('dens_diff df 201', np.sum(np.abs(rho_pyscf - rho_df_201) * coord_weights.numpy())/np.sum(rho_pyscf * coord_weights.numpy()))
    print('dens_diff dfs', np.sum(np.abs(rho_df - rho_df_201) * coord_weights.numpy())/np.sum(rho_df * coord_weights.numpy()))


dens int 9.999995967382509
dens_diff 201 1.3741537882269077e-05
dens_diff df 0.007454463499550962
dens_diff df 201 0.007455531564618262
dens_diff dfs 1.3815889628442003e-05
dens int 9.99999102533892
dens_diff 201 1.2935217806832338e-05
dens_diff df 0.007802019308356851
dens_diff df 201 0.007803018229301732
dens_diff dfs 1.3015480358763593e-05
dens int 10.000002519712746
dens_diff 201 1.336215124054081e-05
dens_diff df 0.007800062641238443
dens_diff df 201 0.0078006192200387086
dens_diff dfs 1.3506375786803609e-05
dens int 10.000001566291278
dens_diff 201 1.2947551865685276e-05
dens_diff df 0.007594009904854091
dens_diff df 201 0.007594878932665895
dens_diff dfs 1.3068292849678793e-05
dens int 9.999998697742338
dens_diff 201 1.490474114161417e-05
dens_diff df 0.007492714311603612
dens_diff df 201 0.007493924958392787
dens_diff dfs 1.5018064533757662e-05
dens int 9.999997966864566
dens_diff 201 1.379566184247119e-05
dens_diff df 0.007461669805968015
dens_diff df 201 0.007462621050859401


In [45]:
print(pyscf_db_orb[0][1]['energy'])

tensor([-76.2408])
